In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import os

print(os.listdir("../data"))

In [ ]:
import os
import pandas as pd

file_path = "../data/customers_features.csv"

if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
    customers = pd.read_csv(file_path)
else:
    print(f"Warning: {file_path} is empty or missing.")
    customers = pd.DataFrame()  # Fallback to an empty DataFrame

In [ ]:
import os

files = [
    "customers_features.csv",
    "orders_features.csv",
    "order_items_features.csv",
    "payments_features.csv",
    "products_features.csv",
    "reviews_features.csv",
    "sellers_features.csv"
]

for file in files:
    path = os.path.join("../data", file)
    print(f"{file:30} {os.path.getsize(path):,} bytes")

In [ ]:
print(os.listdir("../data/processed"))

In [ ]:
processed_files = [
    "customers_features.csv",
    "orders_features.csv",
    "order_items_features.csv",
    "payments_features.csv",
    "products_features.csv",
    "reviews_features.csv",
    "sellers_features.csv",
    "customer_segments.csv",
    "customer_segment_summary.csv"
]

for file in processed_files:
    path = os.path.join("../data/processed", file)
    print(f"{file:35} {os.path.getsize(path):,} bytes")

In [ ]:
from pathlib import Path
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

customers = pd.read_csv("../data/processed/customers_features.csv")
orders_features = pd.read_csv(PROCESSED_DIR / "orders_features.csv")
order_items_features = pd.read_csv(PROCESSED_DIR / "order_items_features.csv")
payments = pd.read_csv("../data/processed/payments_features.csv")
products = pd.read_csv("../data/processed/products_features.csv")
reviews = pd.read_csv("../data/processed/reviews_features.csv")
sellers = pd.read_csv("../data/processed/sellers_features.csv")

# Existing ML segmentation results
customer_segments = pd.read_csv("../data/processed/customer_segments.csv")
customer_segment_summary = pd.read_csv(
    "../data/processed/customer_segment_summary.csv"
)

print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Payments:", payments.shape)
print("Products:", products.shape)
print("Reviews:", reviews.shape)
print("Sellers:", sellers.shape)
print("Customer Segments:", customer_segments.shape)

In [ ]:
print("ORDERS")
print(orders.columns.tolist())

print("\nORDER ITEMS")
print(order_items.columns.tolist())

print("\nPRODUCTS")
print(products.columns.tolist())

print("\nCUSTOMER SEGMENTS")
print(customer_segments.columns.tolist())

print("\nPAYMENTS")
print(payments.columns.tolist())

print("\nSELLERS")
print(sellers.columns.tolist())

In [ ]:
monthly_revenue = (
    orders
    .merge(
        order_items[["order_id", "revenue"]],
        on="order_id",
        how="inner"
    )
    .groupby(["year", "month"], as_index=False)
    ["revenue"]
    .sum()
)

monthly_revenue["date"] = pd.to_datetime(
    monthly_revenue["year"].astype(str) + "-" +
    monthly_revenue["month"].astype(str) + "-01"
)

monthly_revenue = monthly_revenue.sort_values("date")

monthly_revenue.head()

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    monthly_revenue["date"],
    monthly_revenue["revenue"],
    marker="o"
)

plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
category_sales = (
    products[["product_id", "product_category_name"]]
    .merge(
        order_items[["product_id", "price"]],
        on="product_id",
        how="inner"
    )
    .groupby("product_category_name", as_index=False)
    .agg(
        total_sales=("price", "sum"),
        items_sold=("product_id", "count")
    )
    .sort_values("total_sales", ascending=False)
)

top_categories = category_sales.head(10)

top_categories

In [ ]:
plt.figure(figsize=(12, 6))

plt.barh(
    top_categories["product_category_name"][::-1],
    top_categories["total_sales"][::-1]
)

plt.title("Top 10 Product Categories by Sales")
plt.xlabel("Total Sales")
plt.ylabel("Product Category")

plt.tight_layout()
plt.show()

In [ ]:
segment_counts = (
    customer_segments["customer_segment"]
    .value_counts()
    .reset_index()
)

segment_counts.columns = ["customer_segment", "customer_count"]

segment_counts

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    segment_counts["customer_segment"],
    segment_counts["customer_count"]
)

plt.title("Customer Segment Distribution")
plt.xlabel("Customer Segment")
plt.ylabel("Number of Customers")
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
seller_sales = (
    order_items
    .groupby("seller_id", as_index=False)
    .agg(
        total_sales=("price", "sum"),
        items_sold=("order_item_id", "count"),
        total_orders=("order_id", "nunique")
    )
    .sort_values("total_sales", ascending=False)
)

top_sellers = seller_sales.head(10)

top_sellers

In [ ]:
plt.figure(figsize=(12, 6))

plt.barh(
    top_sellers["seller_id"].astype(str)[::-1],
    top_sellers["total_sales"][::-1]
)

plt.title("Top 10 Sellers by Sales")
plt.xlabel("Total Sales")
plt.ylabel("Seller ID")

plt.tight_layout()
plt.show()

In [ ]:
payment_summary = (
    payments
    .groupby("payment_type", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_payment_value=("payment_value", "sum")
    )
    .sort_values("total_payment_value", ascending=False)
)

payment_summary

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    payment_summary["payment_type"],
    payment_summary["total_payment_value"]
)

plt.title("Payment Methods by Total Payment Value")
plt.xlabel("Payment Method")
plt.ylabel("Total Payment Value")

plt.tight_layout()
plt.show()

In [ ]:
order_status = (
    orders["order_status"]
    .value_counts()
    .reset_index()
)

order_status.columns = ["order_status", "order_count"]

order_status

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    order_status["order_status"],
    order_status["order_count"]
)

plt.title("Orders by Status")
plt.xlabel("Order Status")
plt.ylabel("Number of Orders")
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
monthly_revenue.loc[
    monthly_revenue["revenue"].idxmax()
]

In [ ]:
monthly_revenue.loc[
    monthly_revenue["revenue"].idxmin()
]

In [ ]:
first_revenue = monthly_revenue.iloc[0]["revenue"]
last_revenue = monthly_revenue.iloc[-1]["revenue"]

growth = ((last_revenue - first_revenue) / first_revenue) * 100

print(f"Revenue growth: {growth:.2f}%")

<!-- Business insight

Finding: A relatively small group of product categories contributes a substantial portion of total sales.

Evidence: The top-category analysis ranks categories based on total sales.

Business interpretation: High-performing categories can receive greater inventory availability and promotional attention, while lower-performing categories can be investigated for pricing, demand, or product assortment issues. -->

In [ ]:
top_categories

In [ ]:
top_categories["total_sales"].sum()

In [ ]:
top10_sales = top_categories["total_sales"].sum()
total_category_sales = category_sales["total_sales"].sum()

contribution = (top10_sales / total_category_sales) * 100

print(f"Top 10 categories contribute: {contribution:.2f}% of sales")

In [ ]:
customer_segment_summary
customer_segments.groupby("customer_segment").agg(
    customers=("customer_unique_id", "count"),
    avg_spending=("total_spent", "mean"),
    avg_orders=("total_orders", "mean")
).sort_values("avg_spending", ascending=False)

This is especially important because your project already contains customer segmentation using ML.

Business insight

Finding: Customers are divided into distinct behavioral segments based on purchasing activity and spending.

Evidence: The segmentation model groups customers using features such as total spending, order frequency, recency, and customer lifetime.

Business interpretation: Different customer groups can be targeted differently—for example, high-value customers can receive retention-focused offers, while low-frequency customers can be targeted with re-engagement campaigns.

In [ ]:
customer_segments["total_spent"].describe()

In [ ]:
print(
    "Median spending:",
    customer_segments["total_spent"].median()
)

print(
    "Mean spending:",
    customer_segments["total_spent"].mean()
)
# Business insight

# Finding: Customer spending is not evenly distributed.

# Evidence: The spending histogram and descriptive statistics show the distribution of customer-level spending.

# Business interpretation: A small group of high-value customers may contribute disproportionately to revenue, making customer retention and personalized marketing important.

In [ ]:
top_sellers
seller_sales["total_sales"].describe()

top10_seller_sales = top_sellers["total_sales"].sum()
total_seller_sales = seller_sales["total_sales"].sum()

seller_contribution = (
    top10_seller_sales / total_seller_sales
) * 100

print(
    f"Top 10 sellers contribute: {seller_contribution:.2f}% of seller sales"
)
# Business insight

# Finding: Seller performance varies substantially across the marketplace.

# Evidence: Sellers show different sales volumes and numbers of items/orders.

# Business interpretation: The platform can identify high-performing sellers for stronger partnerships while monitoring low-performing sellers for potential operational or assortment issues.

In [ ]:
payment_summary

In [ ]:
payment_summary["total_payment_value"].sum()
payment_summary["percentage"] = (
    payment_summary["total_payment_value"]
    / payment_summary["total_payment_value"].sum()
) * 100

payment_summary
# Business insight

# Finding: Payment methods have different levels of adoption and transaction value.

# Evidence: Payment analysis compares order counts and total payment value across payment types.

# Business interpretation: Understanding payment preferences can help optimize the checkout experience and prioritize reliable payment options.

In [ ]:
order_status

In [ ]:
order_status["percentage"] = (
    order_status["order_count"]
    / order_status["order_count"].sum()
) * 100

order_status
# Business insight

# Finding: The majority of orders belong to specific operational statuses, while a smaller proportion may be cancelled or otherwise unresolved.

# Evidence: Order-status analysis shows the distribution of orders across statuses.

# Business interpretation: Monitoring cancelled or problematic orders can help identify operational issues and improve customer experience.

In [ ]:
print("Average delivery days:",
      orders["delivery_days"].mean())

print("Median delivery days:",
      orders["delivery_days"].median())

print("Late delivery rate:",
      orders["late_delivery"].mean() * 100)


In [ ]:
orders["delivery_delay"].describe()
# Business insight

# Finding: Delivery performance can be quantified using delivery duration and late-delivery indicators.

# Evidence: The dataset contains actual delivery duration, estimated delivery dates, and late-delivery flags.

# Business interpretation: Delivery performance analysis can help identify logistics bottlenecks and improve delivery-time reliability.

In [ ]:
business_insights = pd.DataFrame({
    "Area": [
        "Revenue",
        "Product Categories",
        "Customer Segmentation",
        "Customer Spending",
        "Seller Performance",
        "Payment Methods",
        "Order Status",
        "Delivery Performance"
    ],
    
    "Key Insight": [
        "Revenue varies across months, indicating potential seasonal patterns.",
        "A subset of product categories generates a significant share of sales.",
        "Customers exhibit distinct purchasing and spending behaviors.",
        "Customer spending is unevenly distributed, with some high-value customers.",
        "Seller performance varies significantly across the marketplace.",
        "Payment methods differ in adoption and transaction value.",
        "Orders are distributed across multiple operational statuses.",
        "Delivery duration and late-delivery rates provide measurable logistics indicators."
    ],
    
    "Business Use": [
        "Support revenue forecasting and seasonal planning.",
        "Optimize inventory and promotional strategies.",
        "Enable targeted customer retention and marketing.",
        "Focus retention efforts on high-value customers.",
        "Identify and manage high-performing and underperforming sellers.",
        "Optimize the checkout and payment experience.",
        "Monitor operational issues and cancellation patterns.",
        "Identify opportunities to improve logistics performance."
    ]
})

business_insights

In [ ]:
business_insights.to_csv(
    "../outputs/business_insights.csv",
    index=False
)

print("Business insights saved successfully.")

In [ ]:
import os

os.makedirs("../outputs", exist_ok=True)

In [ ]:
# ==============================
# Dashboard KPIs
# ==============================

total_revenue = order_items["revenue"].sum()

total_orders = orders["order_id"].nunique()

total_customers = customers["customer_id"].nunique()

average_order_value = total_revenue / total_orders

late_delivery_rate = orders["late_delivery"].mean() * 100

print("Total Revenue:", total_revenue)
print("Total Orders:", total_orders)
print("Total Customers:", total_customers)
print("Average Order Value:", average_order_value)
print("Late Delivery Rate:", late_delivery_rate)

In [ ]:
# ==============================
# Dashboard Data
# ==============================

# Monthly Revenue
dashboard_monthly = monthly_revenue.copy()

# Top Categories
dashboard_categories = top_categories.copy()

# Customer Segments
dashboard_segments = segment_counts.copy()

# Payment Methods
dashboard_payments = payment_summary.copy()

# Top Sellers
dashboard_sellers = top_sellers.copy()

# Order Status
dashboard_status = order_status.copy()

In [ ]:
# ==============================
# FINAL BUSINESS DASHBOARD
# ==============================

fig = plt.figure(figsize=(20, 14))

fig.suptitle(
    "E-Commerce Business Analytics Dashboard",
    fontsize=24,
    fontweight="bold"
)

# ------------------------------
# KPI 1
# ------------------------------
ax1 = plt.subplot2grid((4, 3), (0, 0))

ax1.text(
    0.5, 0.6,
    f"₹{total_revenue:,.0f}",
    ha="center",
    va="center",
    fontsize=22,
    fontweight="bold"
)

ax1.text(
    0.5, 0.25,
    "Total Revenue",
    ha="center",
    fontsize=13
)

ax1.axis("off")


# ------------------------------
# KPI 2
# ------------------------------
ax2 = plt.subplot2grid((4, 3), (0, 1))

ax2.text(
    0.5, 0.6,
    f"{total_orders:,}",
    ha="center",
    va="center",
    fontsize=22,
    fontweight="bold"
)

ax2.text(
    0.5, 0.25,
    "Total Orders",
    ha="center",
    fontsize=13
)

ax2.axis("off")


# ------------------------------
# KPI 3
# ------------------------------
ax3 = plt.subplot2grid((4, 3), (0, 2))

ax3.text(
    0.5, 0.6,
    f"{total_customers:,}",
    ha="center",
    va="center",
    fontsize=22,
    fontweight="bold"
)

ax3.text(
    0.5, 0.25,
    "Total Customers",
    ha="center",
    fontsize=13
)

ax3.axis("off")


# ------------------------------
# KPI 4
# ------------------------------
ax4 = plt.subplot2grid((4, 3), (1, 0))

ax4.text(
    0.5, 0.6,
    f"₹{average_order_value:,.0f}",
    ha="center",
    va="center",
    fontsize=22,
    fontweight="bold"
)

ax4.text(
    0.5, 0.25,
    "Average Order Value",
    ha="center",
    fontsize=13
)

ax4.axis("off")


# ------------------------------
# KPI 5
# ------------------------------
ax5 = plt.subplot2grid((4, 3), (1, 1))

ax5.text(
    0.5, 0.6,
    f"{late_delivery_rate:.2f}%",
    ha="center",
    va="center",
    fontsize=22,
    fontweight="bold"
)

ax5.text(
    0.5, 0.25,
    "Late Delivery Rate",
    ha="center",
    fontsize=13
)

ax5.axis("off")


# ------------------------------
# Monthly Revenue
# ------------------------------
ax6 = plt.subplot2grid((4, 3), (1, 2))

ax6.plot(
    dashboard_monthly["date"],
    dashboard_monthly["revenue"],
    marker="o"
)

ax6.set_title("Monthly Revenue")
ax6.set_xlabel("Month")
ax6.set_ylabel("Revenue")
ax6.tick_params(axis="x", rotation=45)


# ------------------------------
# Top Categories
# ------------------------------
ax7 = plt.subplot2grid((4, 3), (2, 0))

ax7.barh(
    dashboard_categories["product_category_name"][::-1],
    dashboard_categories["total_sales"][::-1]
)

ax7.set_title("Top 10 Product Categories")
ax7.set_xlabel("Sales")


# ------------------------------
# Customer Segments
# ------------------------------
ax8 = plt.subplot2grid((4, 3), (2, 1))

ax8.bar(
    dashboard_segments["customer_segment"],
    dashboard_segments["customer_count"]
)

ax8.set_title("Customer Segments")
ax8.set_ylabel("Customers")
ax8.tick_params(axis="x", rotation=30)


# ------------------------------
# Payment Methods
# ------------------------------
ax9 = plt.subplot2grid((4, 3), (2, 2))

ax9.bar(
    dashboard_payments["payment_type"],
    dashboard_payments["total_payment_value"]
)

ax9.set_title("Payment Methods")
ax9.set_ylabel("Payment Value")
ax9.tick_params(axis="x", rotation=30)


# ------------------------------
# Top Sellers
# ------------------------------
ax10 = plt.subplot2grid((4, 3), (3, 0))

ax10.barh(
    dashboard_sellers["seller_id"].astype(str)[::-1],
    dashboard_sellers["total_sales"][::-1]
)

ax10.set_title("Top 10 Sellers")
ax10.set_xlabel("Sales")


# ------------------------------
# Order Status
# ------------------------------
ax11 = plt.subplot2grid((4, 3), (3, 1))

ax11.bar(
    dashboard_status["order_status"],
    dashboard_status["order_count"]
)

ax11.set_title("Order Status")
ax11.set_ylabel("Orders")
ax11.tick_params(axis="x", rotation=30)


# ------------------------------
# Empty final space
# ------------------------------
ax12 = plt.subplot2grid((4, 3), (3, 2))

ax12.axis("off")


plt.tight_layout(rect=[0, 0, 1, 0.96])

plt.show()

In [ ]:
import os

os.makedirs("../outputs", exist_ok=True)

fig.savefig(
    "../outputs/ecommerce_business_dashboard.png",
    dpi=300,
    bbox_inches="tight"
)

print("Dashboard saved successfully.")

In [ ]:
import os

print(os.path.exists("../outputs/ecommerce_business_dashboard.png"))